In [3]:
#ran on hipergator tensorflow 2.18

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets
import tensorflow_io
import librosa

In [4]:
def train(data="training_data_eee4773.npy", labels="training_labels_eee4773.npy", model_path="final_model.keras"):
    
    
    data = np.load(data).T
    t = np.load(labels)

    
    
    X_16k = np.array([librosa.resample(x, orig_sr=48000, target_sr=16000) for x in data])
    
    training_speech, validation_speech = tensorflow_datasets.load("speech_commands", split=["train", "validation"], as_supervised=True)
    
    
    #preprocessing data from the google speech commands
    #it is saved as a tensorflow dataset object so you can preprocess and batch it before passing to model
    def preprocess(audio, label):
        audio = tf.cast(audio, tf.float32) / 32768.0
        audio = tf.reshape(audio, [-1])

        current_length = tf.shape(audio)[0]
        pad_total = tf.maximum(48000 - current_length, 0)
        pad_left = tf.random.uniform([], 0, pad_total + 1, dtype=tf.int32)
        pad_right = pad_total - pad_left

        audio = tf.pad(audio, [[pad_left, pad_right]])
        audio = audio[:48000]

        return audio, label


    training_speech = (
        training_speech
        .map(preprocess)
        .shuffle(1000)
        .batch(64)
    )

    validation_speech = (
        validation_speech
        .map(preprocess)
        .batch(64)
    )

    
    #creating encoder
    conv_encoder = keras.models.Sequential([
        keras.Input(shape=(48000,)),

        keras.layers.MelSpectrogram(fft_length=512,
                                   sequence_stride=160,
                                   sampling_rate=16000,
                                   num_mel_bins=80,
                                   min_freq=80,
                                   power_to_db=True),

        keras.layers.Reshape((80, 301, 1)),

        keras.layers.Conv2D(filters=32,kernel_size=3, padding="same", activation="relu"),
        keras.layers.MaxPool2D(pool_size=2, padding="same"),

        keras.layers.Conv2D(filters=64,kernel_size=3, padding="same", activation="relu"),
        keras.layers.MaxPool2D(pool_size=2, padding="same"),

        keras.layers.Conv2D(filters=128,kernel_size=3, padding="same", activation="relu"),
        keras.layers.MaxPool2D(pool_size=2, padding="same"),
    ])
    
    #training encoder on the google speech dataset
    inp = keras.layers.Input(shape=(48000,))

    x = conv_encoder(inp)
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dropout(0.3)(x)
    x = keras.layers.Dense(500, activation='relu')(x)
    x = keras.layers.Dropout(0.3)(x)
    out = keras.layers.Dense(12, activation="softmax")(x)
    #there is 12 classes in this 



    encoder_classifier = keras.models.Model(inp, out)


    encoder_classifier.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam', metrics=["accuracy"])

    stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    
    history = encoder_classifier.fit(training_speech, validation_data=validation_speech, epochs=10, callbacks=[stopping])
    
    
    #training attached classifier
    for layer in conv_encoder.layers:
            layer.trainable = True
    inp = keras.layers.Input(shape=(48000,))

    x = conv_encoder(inp)
    x = keras.layers.Flatten()(x)

    x = keras.layers.Dropout(0.5)(x)

    x = keras.layers.Dense(32, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001))(x)

    x = keras.layers.Dropout(0.5)(x)
    out = keras.layers.Dense(1, activation='sigmoid')(x)

    clf = keras.models.Model(inp, out)
    
    
    #compiling, running, and returning encoder + attached classifier trained on wake words
    clf.compile(loss='binary_crossentropy',
              optimizer="adam",
              metrics=["accuracy"])

    stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)


    
    #to have balanced validation
    X_train_16k, X_val_16k, t_train, t_val = train_test_split(X_16k, t, test_size=0.15, random_state=0, stratify=t)

    history = clf.fit(X_train_16k, t_train, epochs=30, batch_size=32, validation_data=(X_val_16k, t_val) , callbacks=[stopping])
    
    
    clf.save(model_path)
    return clf

In [5]:
model = train(data = "training_data_eee4773.npy", labels= "training_labels_eee4773.npy", model_path= "training_best_model.keras")

Epoch 1/10


2026-04-22 12:03:19.504940: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2026-04-22 12:03:20.667331: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:376] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


1337/1337 ━━━━━━━━━━━━━━━━━━━━ 223s 166ms/step - accuracy: 0.6383 - loss: 3.5744 - val_accuracy: 0.8149 - val_loss: 0.5592
Epoch 2/10
1337/1337 ━━━━━━━━━━━━━━━━━━━━ 222s 166ms/step - accuracy: 0.7974 - loss: 0.6127 - val_accuracy: 0.8940 - val_loss: 0.3322
Epoch 3/10
1337/1337 ━━━━━━━━━━━━━━━━━━━━ 221s 166ms/step - accuracy: 0.8546 - loss: 0.4455 - val_accuracy: 0.9090 - val_loss: 0.2780
Epoch 4/10
1337/1337 ━━━━━━━━━━━━━━━━━━━━ 222s 166ms/step - accuracy: 0.8780 - loss: 0.3764 - val_accuracy: 0.9236 - val_loss: 0.2447
Epoch 5/10
1337/1337 ━━━━━━━━━━━━━━━━━━━━ 223s 167ms/step - accuracy: 0.8943 - loss: 0.3323 - val_accuracy: 0.9233 - val_loss: 0.2468
Epoch 6/10
1337/1337 ━━━━━━━━━━━━━━━━━━━━ 222s 166ms/step - accuracy: 0.9057 - loss: 0.2973 - val_accuracy: 0.9351 - val_loss: 0.2087
Epoch 7/10
1337/1337 ━━━━━━━━━━━━━━━━━━━━ 225s 168ms/step - accuracy: 0.9096 - loss: 0.2861 - val_accuracy: 0.9367 - val_loss: 0.2031
Epoch 8/10
1337/1337 ━━━━━━━━━━━━━━━━━━━━ 229s 171ms/step - accuracy: 0.9